# Notebook 6: Evaluation on Authentic Marathi (Balbharati Std 6 & 7)

The MarathiMWP training/test data is **translated** from HAWP Hindi, so it may carry
Hindi-influenced phrasing and could inflate the cross-lingual transfer benefit (RQ2).

This notebook evaluates every saved model on a **held-out, manually-sourced authentic
Marathi test set** drawn directly from Maharashtra State Board (Balbharati) Std 6 & 7
mathematics textbooks, and compares accuracy against the translated test set. The
**accuracy gap** (translated - authentic) measures how much performance is attributable
to translation-induced regularities rather than genuine Marathi understanding.

> This set is **never** used for training. It is a pilot out-of-distribution (OOD) probe (n is small;
> report confidence intervals).

In [ ]:
import sys, json, warnings
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path

warnings.filterwarnings('ignore', message='.*use_return_dict.*')
sys.path.insert(0, str(Path('.').resolve()))
from utils.evaluation import compute_metrics, print_metrics, answers_match

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
Path('figures').mkdir(exist_ok=True)

PREFIX = 'generate equation: '
MAX_SRC_LEN, MAX_TGT_LEN = 128, 64

## 1. Load the authentic set and the translated test set

In [ ]:
def load_json(path):
    with open(path, encoding='utf-8') as f:
        return json.load(f)

AUTH_PATH = '../Datasets/marathi/balbharati_authentic_mwp.json'
authentic = load_json(AUTH_PATH)
test      = load_json('splits/marathi_test.json')   # translated test set

# Normalise authentic records to the fields the eval pipeline expects
for r in authentic:
    r.setdefault('Equation', r.get('Equation'))
    r.setdefault('Number of Operators', r.get('Number of Operators'))

print(f'Authentic (Balbharati) : {len(authentic)} problems')
print(f'Translated test set    : {len(test)} problems')

# Sanity: operator-count distribution of the authentic set
from collections import Counter
print('Authentic operator counts:', dict(Counter(r['Number of Operators'] for r in authentic)))
print('Authentic by std        :', dict(Counter(r.get('source','').split()[2] if r.get('source') else '?' for r in authentic)))

## 2. Prediction helper (identical to Notebook 3)

In [ ]:
def generate_predictions(model, tokenizer, data, batch_size=32, prefix=PREFIX):
    model.eval()
    preds = []
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        texts = [prefix + x['Problem'] for x in batch]
        enc   = tokenizer(texts, return_tensors='pt', padding=True,
                          truncation=True, max_length=MAX_SRC_LEN).to(DEVICE)
        with torch.no_grad():
            ids = model.generate(**enc, max_new_tokens=MAX_TGT_LEN, num_beams=4)
        preds.extend(tokenizer.batch_decode(ids, skip_special_tokens=True))
    return preds

## 3. Model registry

Every fine-tuned checkpoint saved by Notebooks 3 & 4. Missing checkpoints are skipped,
so this notebook runs partially before all GPU notebooks are complete and fully afterward.

In [ ]:
MODEL_REGISTRY = [
    # (display name, checkpoint dir, tie_word_embeddings)
    ('mT5-small (Marathi mono)',     'models/mt5_marathi',          False),
    ('IndicBART (Marathi mono)',     'models/indicbart_marathi',    True),
    ('mT5 Zero-shot (Hindi only)',   'models/transfer_zeroshot',    False),
    ('mT5 Few-shot 10% Marathi',     'models/transfer_fewshot_10pct', False),
    ('mT5 Few-shot 25% Marathi',     'models/transfer_fewshot_25pct', False),
    ('mT5 Few-shot 50% Marathi',     'models/transfer_fewshot_50pct', False),
    ('mT5 Multilingual joint',       'models/transfer_multilingual', False),
]
available = [(n, d, t) for (n, d, t) in MODEL_REGISTRY if Path(d).exists()]
print(f'{len(available)}/{len(MODEL_REGISTRY)} checkpoints found:')
for n, d, _ in available:
    print(f'  [x] {n:32s} <- {d}')
for n, d, _ in MODEL_REGISTRY:
    if not Path(d).exists():
        print(f'  [ ] {n:32s} (missing {d} - run Notebooks 3/4 first)')

## 4. Rule-based and template baselines (no GPU, always available)

These deterministic baselines (mirroring Notebook 2, via `utils/baselines.py`) need no
checkpoints, so their rows of Table 5.4 populate immediately. The template baseline is
fit on the Marathi training split.

In [ ]:
from utils.baselines import rule_based_solve, TemplateBaseline

gold_test = [x['Equation'] for x in test]
gold_auth = [x['Equation'] for x in authentic]

train = load_json('splits/marathi_train.json')
template_baseline = TemplateBaseline(train)

BASELINES = [
    ('Rule-Based',     lambda x: rule_based_solve(x)),
    ('Template-Based', lambda x: template_baseline.solve(x)),
]

rows, all_preds = [], {}
for name, solve in BASELINES:
    p_test = [solve(x) for x in test]
    p_auth = [solve(x) for x in authentic]
    all_preds[name] = {'translated': p_test, 'authentic': p_auth}
    m_test = compute_metrics(gold_test, p_test)
    m_auth = compute_metrics(gold_auth, p_auth)
    rows.append({
        'Model'              : name,
        'Translated Acc (%)' : round(m_test['answer_accuracy'], 2),
        'Authentic Acc (%)'  : round(m_auth['answer_accuracy'], 2),
        'Gap (pp)'           : round(m_test['answer_accuracy'] - m_auth['answer_accuracy'], 2),
        'Auth Eq Acc (%)'    : round(m_auth['equation_accuracy'], 2),
        'Auth BLEU'          : round(m_auth['bleu'], 2),
    })
    print(f'{name:32s} translated={m_test["answer_accuracy"]:5.2f}%  '
          f'authentic={m_auth["answer_accuracy"]:5.2f}%  '
          f'gap={m_test["answer_accuracy"]-m_auth["answer_accuracy"]:+5.2f}pp')

## 5. Evaluate each neural model on translated vs. authentic

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

for name, ckpt, tie in available:
    tok   = AutoTokenizer.from_pretrained(ckpt)
    model = AutoModelForSeq2SeqLM.from_pretrained(ckpt, tie_word_embeddings=tie).to(DEVICE)

    p_test = generate_predictions(model, tok, test)
    p_auth = generate_predictions(model, tok, authentic)
    all_preds[name] = {'translated': p_test, 'authentic': p_auth}

    m_test = compute_metrics(gold_test, p_test)
    m_auth = compute_metrics(gold_auth, p_auth)
    rows.append({
        'Model'              : name,
        'Translated Acc (%)' : round(m_test['answer_accuracy'], 2),
        'Authentic Acc (%)'  : round(m_auth['answer_accuracy'], 2),
        'Gap (pp)'           : round(m_test['answer_accuracy'] - m_auth['answer_accuracy'], 2),
        'Auth Eq Acc (%)'    : round(m_auth['equation_accuracy'], 2),
        'Auth BLEU'          : round(m_auth['bleu'], 2),
    })
    print(f'{name:32s} translated={m_test["answer_accuracy"]:5.2f}%  '
          f'authentic={m_auth["answer_accuracy"]:5.2f}%  '
          f'gap={m_test["answer_accuracy"]-m_auth["answer_accuracy"]:+5.2f}pp')
    del model
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

results = pd.DataFrame(rows)
results

## 6. Wilson 95% confidence interval on authentic accuracy (n is small)

In [ ]:
def wilson_ci(k, n, z=1.96):
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2*n)) / denom
    half = (z * ((p*(1-p)/n + z**2/(4*n**2)) ** 0.5)) / denom
    return (round(100*(centre-half), 1), round(100*(centre+half), 1))

n_auth = len(authentic)
for name in all_preds:
    p_auth = all_preds[name]['authentic']
    k = sum(answers_match(g, p) for g, p in zip(gold_auth, p_auth))
    lo, hi = wilson_ci(k, n_auth)
    print(f'{name:32s} {100*k/n_auth:5.2f}%  (95% CI [{lo}, {hi}], {k}/{n_auth})')

## 7. Figure: translated vs. authentic accuracy (the transfer-artefact gap)

In [ ]:
if not results.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(len(results))
    w = 0.38
    ax.bar(x - w/2, results['Translated Acc (%)'], w, label='Translated test (HAWP-derived)')
    ax.bar(x + w/2, results['Authentic Acc (%)'],  w, label='Authentic test (Balbharati)')
    ax.set_xticks(x)
    ax.set_xticklabels(results['Model'], rotation=30, ha='right', fontsize=8)
    ax.set_ylabel('Answer Accuracy (%)')
    ax.set_title('Translated vs. Authentic Marathi Accuracy by Model')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('figures/fig_5_authentic_gap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: figures/fig_5_authentic_gap.png')

## 8. Per-operation accuracy on the authentic set (best model)

In [ ]:
from utils.data_utils import classify_operation

if all_preds:
    best = max(all_preds, key=lambda nm:
               sum(answers_match(g, p) for g, p in zip(gold_auth, all_preds[nm]['authentic'])))
    p_auth = all_preds[best]['authentic']
    by_op = {}
    for rec, pred in zip(authentic, p_auth):
        op = rec.get('operation_type') or classify_operation(rec['Equation'])
        d = by_op.setdefault(op, [0, 0])
        d[1] += 1
        d[0] += int(answers_match(rec['Equation'], pred))
    print(f'Best model on authentic set: {best}\n')
    for op, (c, t) in sorted(by_op.items()):
        print(f'  {op:15s} {100*c/t:5.1f}%  ({c}/{t})')

## 9. Save predictions and results table

In [ ]:
if not results.empty:
    results.to_csv('figures/table_5_4_authentic_eval.csv', index=False)
    print('Saved: figures/table_5_4_authentic_eval.csv')

    out = []
    for i, rec in enumerate(authentic):
        row = {'pIndex': rec['pIndex'], 'Problem': rec['Problem'],
               'Gold': rec['Equation'], 'source': rec.get('source')}
        for name in all_preds:
            row[name] = all_preds[name]['authentic'][i]
        out.append(row)
    with open('splits/authentic_predictions.json', 'w', encoding='utf-8') as f:
        json.dump(out, f, ensure_ascii=False, indent=2)
    print('Saved: splits/authentic_predictions.json')
else:
    print('No models evaluated yet - run Notebooks 3 & 4 to produce checkpoints, then re-run.')